# 08: QAT with onebitllms -> smelt inference

Train a ternary model with [onebitllms](https://github.com/tiiuae/onebitllms)
(STE + Triton kernels), then pack for fast CPU inference with smelt.

| step | tool | what |
|:---|:---|:---|
| replace linears | onebitllms | BitNetLinear with STE + per-block scaling |
| train | trl SFTTrainer | standard HF training loop on GPU |
| quantize to 1-bit | onebitllms | freeze ternary weights |
| pack for CPU inference | smelt.quantize | TL1 packing |

Requires: `pip install onebitllms trl datasets`

In [ ]:
import os
import sys

if "COLAB_GPU" in os.environ:
    os.system("pip install -q git+https://github.com/PritRaj1/smelt.git")
    os.system("pip install -q 'transformers>=4.51,<5' triton onebitllms trl datasets")
else:
    sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd())))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

## Train with onebitllms

In [2]:
from onebitllms import replace_linear_with_bitnet_linear

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto")

model = replace_linear_with_bitnet_linear(model)
print("BitNetLinear layers inserted")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BitNetLinear layers inserted


In [3]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:1000]")

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="/tmp/smelt_qat",
        num_train_epochs=1,
        learning_rate=1e-4,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        logging_steps=10,
        bf16=True,
        max_length=256,
    ),
)
trainer.train()
trainer.save_model("/tmp/smelt_qat")

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
In file included from /usr/include/python3.14/Python.h:14,
                 from /tmp/tmpr_u3u16i/__triton_launcher.c:7:
/usr/include/python3.14/pyconfig.h:2007:9: warning: ‘_POSIX_C_SOURCE’ redefined
 2007 | #define _POSIX_C_SOURCE 200809L
      |         ^~~~~~~~~~~~~~~
In file included from /usr/include/bits/libc-header-start.h:33,
                 from /usr/include/stdlib.h:26,
                 from /home/pritmanguy/Work/smelt/.venv/lib/python3.14/site-packages/triton/backends/nvidia/include/cuda.h:56,
                 from /tmp/tmpr_u3u16i/__triton_launcher.c:2:
/usr/include/features.h:319:10: note: this is the location of the previous definition
  319 | # define _POSIX_C_SOURCE        202405L
      |          ^~~~~~~~~~~~~~~
In file included

OutOfMemoryError: CUDA out of memory. Tried to allocate 22.00 MiB. GPU 0 has a total capacity of 3.63 GiB of which 23.81 MiB is free. Process 1558 has 67.89 MiB memory in use. Process 161923 has 56.00 MiB memory in use. Including non-PyTorch memory, this process has 3.48 GiB memory in use. Of the allocated memory 3.36 GiB is allocated by PyTorch, and 37.75 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
from onebitllms import quantize_to_1bit

quantize_to_1bit("/tmp/smelt_qat", "/tmp/smelt_qat_1bit")
print("quantized to 1-bit")

## Pack for smelt inference

In [ ]:
import smelt

model = AutoModelForCausalLM.from_pretrained("/tmp/smelt_qat_1bit", dtype=torch.float32)
model.eval()
smelt.quantize(model)
print("packed for CPU inference")

## Generate

In [ ]:
prompts = [
    "The capital of UK is",
    "def fibonacci(n):",
    "Explain x86 AVX2 in simple terms:",
]

for p in prompts:
    ids = tokenizer.encode(p, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=32, do_sample=False)
    print(tokenizer.decode(out[0], skip_special_tokens=True))
    print()